# W6D3 — Agentic AI: Build a ReAct Tool-Using Agent from Scratch — Guided

**Week 6 · Day 3 · LLMs** · Lab

> **The big idea.** A language model answers; an **agent** acts. The step from one
> to the other is a loop — a short Python loop — that asks the model to pick a
> **tool**, runs that tool, feeds the result back in, and asks again until the
> model says it's done. Everything else — LangGraph, CrewAI, OpenAI Agents SDK —
> is a nicer wrapper around that same loop.

In **D2** you combined an LLM with an embedder to ground answers in documents.
Today you combine an LLM with **arbitrary Python functions** to let it take
actions. By the end you will have built a working **ReAct agent** in about 60
lines of code — no framework, no dependency, just the Qwen chat model from
yesterday plus two tool functions.

<div dir="rtl" align="right">

> **الفكرة الكبرى.** النموذج اللغويّ يُجيب؛ **الوكيل** يتصرّف. الخطوة بينهما حلقة
> — حلقة Python قصيرة — تطلب من النموذج اختيار **أداة**، تُشغّلها، تُرجع النتيجة
> إليه، وتسأله ثانيةً حتى يقول "انتهيت". كلّ ما سوى ذلك — LangGraph وCrewAI و
> OpenAI Agents SDK — مجرّد غلاف أنيق لنفس الحلقة.

في **D2** جمعت LLM مع مُضمِّن لترسيخ الإجابات في وثائق. اليوم تجمع LLM مع **دوال
Python** لتجعله يتصرّف. في النهاية ستكون قد بنيت **وكيل ReAct** عاملًا في نحو ٦٠
سطرًا، بلا إطار ولا تبعيّة.

</div>

> **This is the guided version.** Most of the code is already here. Fill in the lines marked
> `# TODO`. If you want the full challenge, use the `_blank` version instead.

<div dir="rtl" align="right">

> **هذه النسخة الموجَّهة.** معظم الشيفرة موجودة، وعليك إكمال الأسطر المعلَّمة بـ `# TODO`.
> وإذا أردت التحدّي الكامل فاستخدم نسخة `_blank`.

</div>

## Learning objectives

Concretely, you can:

- Explain what makes an LLM into an **agent**: tools, a loop, and the ability to
  decide which tool to use next.
- Define a Python function as a **tool** the LLM can call, including a plain-text
  specification the LLM reads at prompt-time.
- Implement one iteration of the **ReAct** pattern: parse the model's Thought +
  Action, execute the tool, format an Observation, and feed everything back.
- Wrap the single step in a loop with a **max-steps safety cap** so a confused
  agent cannot run forever.
- Explain the two main failure modes of a tool-using agent: (a) hallucinating a
  tool call the agent doesn't have, (b) looping forever on an impossible task.

<div dir="rtl" align="right">

## الأهداف التعلّمية

- أن تشرح ما الذي يجعل من LLM **وكيلًا**: أدوات، حلقة، وقدرة على اختيار الأداة.
- أن تُعرِّف دالّة Python كـ**أداة** يستطيع النموذج استدعاءها، مع مواصفات نصّيّة
  يقرأها عند بناء المُحثّ.
- أن تُنفّذ تكرارًا واحدًا لنمط **ReAct**: parse the Thought + Action، شغّل الأداة،
  صِغ Observation، وأعد الكلّ.
- أن تلفّ الخطوة الواحدة بحلقة مع **حدّ أقصى للخطوات** حتى لا يدور الوكيل أبدًا.
- أن تشرح وضعَي الفشل الرئيسيَّين: (أ) هلوسة أداة غير موجودة، (ب) الدوران الأبديّ
  في مهمّة مستحيلة.

</div>


## About the model and data

**Model:** `Qwen/Qwen2.5-1.5B-Instruct` — 1.5B parameters, instruction-tuned,
reliable at following structured output patterns like ReAct. Fits in ~3 GB RAM,
runs on Colab T4 in seconds per step, and on Colab CPU in a few seconds per
step. We use a bigger model than D2 (1.5B vs 0.5B) because reliable
**structured output following** is harder than one-shot answering — the agent
asks the model to emit a specific `Thought/Action/Action Input` format on
every step, and 0.5B gets it wrong too often.

**Data (tools and the world they operate on):** instead of a corpus, the agent
has access to:

- A **calculator** tool that safely evaluates arithmetic expressions
  (`"40 * 2500 / 3.75"`).
- A **tiny knowledge base** of 5 fictional facts about Olo Academy
  (cohort size, tuition, exchange rate, program length, lead instructor) — a
  Python `dict` with fuzzy string matching.

Together these two tools let the agent answer compositional questions like
"what is the total AIEP tuition revenue in USD?" that require **both** a
factual lookup **and** a calculation. In production this is where you would
plug in: a real search engine, a vector store (D2's RAG!), a SQL query runner,
an HTTP client, a code sandbox — the agent loop is identical regardless.

<div dir="rtl" align="right">

## عن النموذج والبيانات

**النموذج:** `Qwen/Qwen2.5-1.5B-Instruct` — ١٫٥ مليار معامل، مُدرَّب على
التعليمات، موثوق في اتّباع صيغ المخرجات مثل ReAct. استخدمنا نموذجًا أكبر من D2
لأن اتّباع الصيغة بثبات أصعب من إجابة واحدة.

**البيانات (الأدوات والعالم الذي تعمل فيه):** بدلًا من مجموعة نصوص، يملك
الوكيل:

- أداة **حاسبة** تحسب التعابير بأمان (`"40 * 2500 / 3.75"`).
- **قاعدة معرفة صغيرة** من ٥ حقائق خياليّة عن أكاديمية أولو — `dict` بـPython
  مع مطابقة فضفاضة.

تمكّن الأداتان الوكيل من الإجابة عن أسئلة مركّبة مثل "ما إجمالي إيرادات AIEP
بالدولار؟" التي تحتاج **بحثًا** و**حسبة**. في الإنتاج هنا تضع محرّك بحث حقيقي،
مخزن متّجهات (RAG من D2!)، منفّذ SQL، عميل HTTP، أو صندوق كود — حلقة الوكيل
نفسها.

</div>


## Setup

<div dir="rtl" align="right">

## الإعداد

</div>


In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
import sys, subprocess
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "transformers>=4.40", "accelerate"])

import re, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 42
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch={torch.__version__}  device={DEVICE}")

In [ ]:
# === Load the chat LLM ==============================================================
LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm       = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float32).to(DEVICE)
llm.eval()

@torch.no_grad()
def chat(system_message, user_message, max_new_tokens=200, stop=None):
    msgs = [{"role":"system","content":system_message},
            {"role":"user",  "content":user_message}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(prompt, return_tensors="pt").to(llm.device)
    out = llm.generate(
        **enc, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    if stop:
        for s in stop:
            if s in text:
                text = text[:text.index(s)]
    return text.strip()

print(f"Loaded {LLM_NAME}: {sum(p.numel() for p in llm.parameters())/1e6:.0f}M params")

## Section 1 — Warm-up: why a plain LLM is not enough  *(~10 min)*

Nothing to implement. Watch two things:

1. A plain LLM try (and quietly fail) to answer a question that needs **both**
   fresh math and a factual lookup. It confidently makes something up.
2. A **manual** tool call: you compute the right answer with Python, hand it to
   the LLM as context, and the LLM composes the final sentence.

The agent in Section 2 just automates step 2 so the LLM, not you, decides when
to call the tool.

<div dir="rtl" align="right">

## القسم ١ — إحماء: لماذا LLM وحده لا يكفي  *(≈١٠ دقائق)*

لا شيء تنفّذه. راقب شيئين:

١. نموذج يحاول (ويفشل بثقة) إجابة سؤال يحتاج حسابًا طازجًا وبحثًا معلوماتيًّا.
٢. استدعاء **يدويّ** لأداة: تحسب الإجابة بـPython، تسلّمها للنموذج، فيُركّب الجملة
   النهائيّة.

وكيل القسم ٢ يُؤتمت الخطوة ٢، فيقرّر النموذج (لا أنت) متى يستدعي الأداة.

</div>


In [ ]:
# === Primitive 1: a plain LLM, no tools ============================================
Q = ("If Olo Academy's AIEP cohort has 40 students and tuition were 2500 SAR each, "
     "what is the total revenue in US dollars at 3.75 SAR per USD?")
print("Q:", Q)
print()
print("LLM alone (no tools):")
print(chat(system_message="You are a helpful assistant.", user_message=Q, max_new_tokens=150))

In [ ]:
# === Primitive 2: manual tool call ==================================================
# You do the math in Python, then tell the LLM the result and ask it to answer.
calc_result = (40 * 2500) / 3.75   # SAR -> USD
print(f"Python computed: {calc_result:.2f} USD")
print()
print("LLM with the fact handed to it:")
print(chat(
    system_message="You are a helpful assistant. Use ONLY the facts given to you.",
    user_message=f"Fact: 40 students * 2500 SAR / (3.75 SAR per USD) = {calc_result:.2f} USD.\n\n"
                 f"Question: {Q}",
    max_new_tokens=80,
))

The LLM is a lot more useful when a computation it cannot reliably do has
already been done. In Section 2 we let it **ask for the computation** instead
of us hard-coding it.

<div dir="rtl" align="right">

النموذج أكثر فائدة عند تسليمه حسبةً لا يُجيدها. في القسم ٢ نجعله **يطلب** الحسبة
بدلًا من أن نُجريها بأنفسنا.

</div>


## Section 2 — Build the agent  *(~55 min)*

Five tasks. Each one new piece of the agent loop:

1. **A calculator tool** — a plain Python function the LLM can call for math.
2. **A knowledge-base tool** — a tiny dict lookup that stands in for a search
   engine or vector store.
3. **The ReAct system prompt** — the instructions that teach the LLM how to call
   those tools and in what format.
4. **One agent step** — call the LLM, parse `Action: ...` / `Action Input: ...`,
   run the tool, format the Observation.
5. **The loop** — run the step up to `max_steps` times until the model emits
   `Final Answer: ...`.

<div dir="rtl" align="right">

## القسم ٢ — ابنِ الوكيل  *(≈٥٥ دقيقة)*

خمس مهام. كلّ واحدة جزء جديد من حلقة الوكيل:

١. **أداة حاسبة** — دالّة Python عاديّة.
٢. **أداة قاعدة معرفة** — بحث في dict صغير.
٣. **مُحثّ ReAct** — تعليمات تعلّم النموذج كيف يستدعي الأدوات وبأيّ صيغة.
٤. **خطوة وكيل واحدة** — نادِ LLM، parse `Action:` / `Action Input:`، شغّل الأداة،
   صِغ Observation.
٥. **الحلقة** — شغّل الخطوة حتى `max_steps` أو حتى يقول `Final Answer: ...`.

</div>


### Task 1 — Build a calculator tool

Write `tool_calculator(expression)` that evaluates a simple arithmetic
expression and returns the result as a string. For safety, restrict allowed
characters to digits, operators (`+ - * / . ( ) %`), and spaces. Anything else
should return an error string the agent can read back.

<div dir="rtl" align="right">

### المهمّة ١ — ابنِ أداة حاسبة

اكتب `tool_calculator(expression)` التي تحسب تعبيرًا حسابيًّا وتُرجع النتيجة
كنصّ. للسلامة، اقتصر على الأرقام والمعاملات والأقواس والفراغات.

</div>


In [ ]:

def tool_calculator(expression: str) -> str:
    # TODO: Implement tool_calculator(expression) returning a string result or a short error.
    # مهمة: نفّذ tool_calculator(expression) تُرجع نتيجة كنصّ أو خطأً قصيرًا.

for e in ["2 + 2", "(40 * 2500) / 3.75", "sqrt(16)", "100 % 7"]:
    print(f"calc({e!r:>22}) -> {tool_calculator(e)}")

### Task 2 — Build a knowledge-base lookup tool

Write `tool_kb_lookup(key)` that looks up `key` in the fictional handbook
below. Return the matching fact as a string, or a short "not found" message
if the key is absent. This stands in for a real search engine, vector DB, or
RAG pipeline — the agent doesn't need to know which.

<div dir="rtl" align="right">

### المهمّة ٢ — أداة بحث في قاعدة معرفة

اكتب `tool_kb_lookup(key)` تبحث عن `key` في الدليل الخياليّ أدناه. أرجع الحقيقة
المطابقة أو رسالة "لم يوجد".

</div>


In [ ]:

KB = {
    "cohort size":        "The AIEP cohort of October 2026 has 40 students.",
    "tuition per student":"Tuition is 2500 SAR per student.",
    "USD exchange rate":  "The current exchange rate is 3.75 SAR per 1 USD.",
    "program length":     "The AIEP bootcamp runs for 8 weeks.",
    "lead instructor":    "The lead instructor is Ghaida Alshehri.",
}

def tool_kb_lookup(key: str) -> str:
    # TODO: Implement tool_kb_lookup(key) that returns a matching fact or a not-found message.
    # مهمة: نفّذ tool_kb_lookup(key) تُرجع حقيقة مطابقة أو رسالة "لم يوجد".

for k in ["cohort size", "exchange rate", "capstone date"]:
    print(f"kb({k!r:>22}) -> {tool_kb_lookup(k)}")

### Task 3 — Write the ReAct system prompt

The LLM only knows how to use your tools if the system prompt **tells it** —
their names, what they do, and the exact output format it must produce on each
turn.

Fill in `SYSTEM_PROMPT` with:

- A one-line role statement.
- A **Tools** section that names each tool, its purpose, and one example input.
- A **Format** section showing the exact pattern:
  ```
  Thought: <reasoning>
  Action: <one of calculator | kb_lookup>
  Action Input: <string input for the tool>
  ```
  …followed by a note that the Observation will be filled in by the system,
  and that when the agent is done it should emit:
  ```
  Thought: <reasoning>
  Final Answer: <the answer for the user>
  ```

<div dir="rtl" align="right">

### المهمّة ٣ — اكتب مُحثّ ReAct

النموذج يعرف أدواتك فقط إذا **أخبره** المُحثّ. اكتب `SYSTEM_PROMPT` فيه: دور
النموذج، قسم Tools بأسماء ومواصفات، وقسم Format بالصيغة الدقيقة.

</div>


In [ ]:

# TODO: Write the SYSTEM_PROMPT that teaches the LLM how to use your two tools in ReAct format.
# مهمة: اكتب SYSTEM_PROMPT يعلّم النموذج استخدام الأداتين بصيغة ReAct.

print(SYSTEM_PROMPT[:600], "...")

### Task 4 — Implement one agent step

Write `agent_step(scratchpad)` that:

1. Calls `chat(SYSTEM_PROMPT, scratchpad, stop=["Observation:"])` to get the
   model's next move (and stop before any observation it might hallucinate).
2. Returns `("final", <text>)` if the output contains `Final Answer:`.
3. Otherwise parses `Action:` and `Action Input:` lines, runs the matching tool,
   and returns `("tool", <llm_output + Observation line>)` so the scratchpad can
   be extended.

<div dir="rtl" align="right">

### المهمّة ٤ — نفّذ خطوة وكيل واحدة

اكتب `agent_step(scratchpad)` تنادي chat، تطبع، تُحلِّل Action/Action Input،
تُشغّل الأداة المناسبة، وتُرجع الاستكمال الذي سيُضاف إلى scratchpad.

</div>


In [ ]:
# === Helper: parse an Action from the LLM's text ===================================
def parse_action(text):
    """Return (action, action_input) parsed from the LLM text, or (None, None)."""
    m_act = re.search(r"Action\s*:\s*(\S+)", text)
    m_in  = re.search(r"Action Input\s*:\s*(.+)", text)
    if not m_act or not m_in:
        return None, None
    return m_act.group(1).strip(), m_in.group(1).strip().strip('"\'`')

TOOLS = {"calculator": tool_calculator, "kb_lookup": tool_kb_lookup}

In [ ]:

def agent_step(scratchpad: str):
    # TODO: Implement one step of the ReAct loop. Return ("final", answer) or ("tool", augmented).
    # مهمة: نفّذ خطوة ReAct واحدة. أرجع ("final", answer) أو ("tool", augmented).

### Task 5 — Loop until Final Answer, with a safety cap

Wrap `agent_step` in a loop. Start the scratchpad with `"Question: <q>\n"`.
Call `agent_step(scratchpad)` up to `max_steps` times. On each iteration print
the model's output so you can watch the agent think. Stop when `agent_step`
returns `("final", answer)`, or when you hit `max_steps` (return `"[max steps
reached]"`).

Then test it on a question that **requires both tools**: look up a fact, do a
calculation with it, answer in natural language.

<div dir="rtl" align="right">

### المهمّة ٥ — الحلقة حتى Final Answer مع حدّ أقصى

لفّ `agent_step` بحلقة. ابدأ scratchpad بـ`"Question: <q>\n"`. اطبع كل خطوة
لمراقبة التفكير. توقّف عند الوصول إلى الإجابة أو max_steps.

ثم اختبره على سؤال يحتاج **كلتا الأداتين**: ابحث عن حقيقة، احسب بها، أجب
بلغة طبيعيّة.

</div>


In [ ]:

def run_agent(question: str, max_steps: int = 5) -> str:
    # TODO: Implement run_agent(question, max_steps=5). Print each step; return the final answer.
    # مهمة: نفّذ run_agent(question, max_steps=5). اطبع كل خطوة؛ أرجع الإجابة النهائيّة.

QUESTION = ("How much total tuition revenue in US dollars does the current "
            "AIEP cohort generate, given the handbook's tuition and current rate?")
print("Q:", QUESTION, "\n")
final = run_agent(QUESTION, max_steps=6)

### Takeaway

An agent is just a tiny loop around an LLM. Three things make it work:

1. A **system prompt** that teaches the model the allowed tools and the exact
   output format.
2. A **parser** that extracts the Action from the model's output.
3. A **tool runner** that executes real Python and formats the Observation back
   into text the model can read next turn.

LangGraph, CrewAI, OpenAI Agents SDK, LangChain Agents — all of them wrap this
same pattern with better ergonomics (typed tool specs, retries, tracing,
multi-agent orchestration). But the core is what you just wrote.

<div dir="rtl" align="right">

### الخلاصة

الوكيل حلقة صغيرة حول LLM. ثلاث قطع تجعله يعمل: مُحثّ نظام يُعلّم النموذج
الأدوات والصيغة، parser يستخرج Action من المخرج، ومُشغّل أدوات يُشغّل Python
الحقيقيّ ويُعيد Observation. كلّ الأُطر الكبيرة تلفّ هذا النمط فقط.

</div>


## Section 3 — Rebuild the agent with LangGraph  *(~35 min)*

Section 2 taught you what an agent is by making you write every line. In the
real world you will reach for a framework instead. The most common one in
production (as of late 2026) is **LangGraph** — same ReAct loop you just
built, exposed as a **graph of nodes** you wire together.

In this section you re-implement the exact same agent — same `chat()`, same
tools, same system prompt — but as a LangGraph `StateGraph`. By the end you
can read any LangGraph tutorial, PR, or production codebase and know what
each piece maps to in your from-scratch version:

| Section 2 (scratch) | Section 3 (LangGraph) |
|---|---|
| `scratchpad` string | `AgentState` TypedDict |
| one iteration of the loop | a **node** function |
| the `if "Final Answer:" in ...` check | a **conditional edge** |
| `TOOLS[action](action_input)` | a **tool node** |
| `while step < max_steps:` | `graph.compile()` + `.invoke()` |

Three tasks: (1) define the graph state + two nodes, (2) wire them together
and compile, (3) run on the same questions and verify behaviour matches.

<div dir="rtl" align="right">

## القسم ٣ — أعِد بناء الوكيل بـLangGraph  *(≈٣٥ دقيقة)*

القسم ٢ علّمك ما هو الوكيل بأن جعلك تكتب كلّ سطر. في العالم الحقيقيّ ستلجأ إلى
إطار. الأشهر إنتاجيًّا (في أواخر ٢٠٢٦) هو **LangGraph** — نفس حلقة ReAct التي
بنيتها، لكن ككـ**مُخطَّط من العُقَد**.

تُعيد في هذا القسم بناء نفس الوكيل — بنفس `chat()`، ونفس الأدوات، ونفس
المُحثّ — لكن كـ`StateGraph`. ستتمكّن من قراءة أيّ درس أو PR أو شيفرة إنتاج
بـLangGraph ومعرفة ما يقابل كلّ قطعة في نسختك من الصفر.

ثلاث مهام: (١) عرِّف حالة المُخطَّط + عُقدتين، (٢) اربطهما وترجم، (٣) شغّل على
نفس الأسئلة وتحقّق أن السلوك مطابق.

</div>


In [ ]:
# === Install LangGraph (Colab) and import ==========================================
import sys, subprocess
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "langgraph>=0.2"])

from typing import TypedDict
from langgraph.graph import StateGraph, END

print("LangGraph imported.")

### Task 1 — Define the graph state and the two nodes

A LangGraph **state** is a `TypedDict` that flows through the graph. Each
node reads the state, computes something, and returns a dict of fields to
merge back.

For our agent the state has exactly three fields:

- `question`      — the user's original question (never changes).
- `scratchpad`    — the running transcript (grows every step).
- `last_output`   — what the LLM said in the most recent think step.

Then implement two nodes:

- `think_node(state)` — call `chat(SYSTEM_PROMPT, state["scratchpad"], stop=["Observation:"])`,
  append the output to the scratchpad, record `last_output`.
- `tool_node(state)`  — reuse `parse_action()` and the `TOOLS` dict from
  Section 2 to run the tool, then append `Observation: ...\n` to the scratchpad.

Both nodes return a *patch* dict with only the fields they changed.

<div dir="rtl" align="right">

### المهمّة ١ — عرِّف حالة المُخطَّط والعُقدتين

**حالة** LangGraph هي `TypedDict` تجري عبر المُخطَّط. كل عُقدة تقرأ الحالة،
تحسب شيئًا، وتُرجع dict بالحقول التي تغيّرت فقط.

الحالة لدينا بثلاثة حقول: `question`، `scratchpad`، `last_output`. ثم
`think_node` ينادي chat() ويُحدِّث scratchpad، و`tool_node` يُحلّل الفعل ويُشغّل
الأداة ويلحق Observation. كلتاهما تُعيدان dict جزئيًّا.

</div>


In [ ]:

# TODO: Define AgentState (TypedDict) and the two node functions.
# مهمة: عرِّف AgentState (TypedDict) ودالّتَي العُقدتين.

print("AgentState and nodes defined.")

### Task 2 — Build the graph: nodes, edges, and the conditional edge

Wire the two nodes together:

- `think` is the **entry point** (where every run starts).
- From `think`, **conditionally** go to `END` if the output contains
  `Final Answer:`, otherwise go to `tool`.
- From `tool`, **unconditionally** go back to `think`.

That is it. The compiled graph is a callable: `agent.invoke({"question": ..., "scratchpad": ..., "last_output": ""})` returns the final state.

<div dir="rtl" align="right">

### المهمّة ٢ — ابنِ المُخطَّط: عُقَد، حوافّ، وحافّة شرطيّة

اربط العُقدتين: `think` هو مدخل كلّ تشغيل. من `think` اذهب شرطيًّا إلى `END`
إن احتوى المخرج على `Final Answer:`، وإلّا إلى `tool`. من `tool` ارجع
دائمًا إلى `think`. المُخطَّط المُترجَم قابل للنداء بـ`.invoke(...)`.

</div>


In [ ]:

# TODO: Build the StateGraph, add conditional edge from think, and compile.
# مهمة: ابنِ StateGraph، أضف حافّة شرطيّة من think، وترجم.

print("Graph compiled. Nodes:", list(builder.nodes))

### Task 3 — Run the LangGraph agent and compare behaviour

Invoke the compiled agent on the **same multi-tool question** from Section 2,
and then on an **out-of-corpus question** that should fall back to a sensible
"I don't know".

Pass `config={"recursion_limit": 12}` so LangGraph refuses to loop forever —
the equivalent of your `max_steps` safety cap in Section 2.

Print the final answer **and** the full scratchpad so you can see the agent
loop played out as before — just orchestrated by LangGraph now.

<div dir="rtl" align="right">

### المهمّة ٣ — شغّل وكيل LangGraph وقارن السلوك

نادِ الوكيل المُترجَم على **نفس السؤال متعدّد الأدوات** من القسم ٢، ثم على
**سؤال خارج المجموعة**.

مرّر `config={"recursion_limit": 12}` حتى يرفض LangGraph الدوران إلى الأبد —
المكافئ لـ`max_steps` في القسم ٢.

اطبع الإجابة النهائيّة **و** scratchpad كاملًا لترى الحلقة التي بنيتها بالأمس
تجري الآن تحت إدارة LangGraph.

</div>


In [ ]:

QUESTIONS = [
    "How much total tuition revenue in US dollars does the current AIEP cohort "
    "generate, given the handbook's tuition and current rate?",
    "What is the live price of Bitcoin in Saudi Riyals right now?",
]

# TODO: Invoke the agent on two questions, print the final answer and the scratchpad for each.
# مهمة: نادِ الوكيل على سؤالين، اطبع الإجابة وscratchpad لكلٍّ منهما.

### Write-up

In **one English sentence**, write down when you would reach for LangGraph
over the from-scratch loop you built in Section 2. (Hint: think about
multi-agent systems, state persistence across turns, streaming, tracing, and
whether your team has already picked a stack.)

> *Example answer: "The from-scratch loop is better for learning, prototyping,
> and single-agent tasks where you want zero dependencies; LangGraph starts to
> pay for itself as soon as you need multi-agent orchestration, persistent
> state across sessions, or integration with the rest of the LangChain
> ecosystem (tracing, streaming, retries)."*

<div dir="rtl" align="right">

### كتابة

في **جملة إنجليزيّة واحدة**، متى تلجأ إلى LangGraph بدلًا من الحلقة من
الصفر؟ (تلميح: فكّر في الأنظمة متعدّدة الوكلاء، استمرار الحالة بين الجلسات،
التدفّق، التتبّع، وما إذا كان فريقك قد اختار منصّة بالفعل.)

</div>


## Wrap-up

Four things you can now do:

1. **Build a ReAct agent from scratch** in ~60 lines of pure Python, no framework.
2. **Rebuild the same agent as a LangGraph** `StateGraph` with explicit nodes,
   edges, and a conditional edge — and know what every piece corresponds to
   in your scratch version.
3. **Explain** why bigger-than-D2 models are needed for reliable structured
   output, and spot the two main failure modes (hallucinated tool, infinite
   loop) — plus the safety mechanisms (`max_steps`, `recursion_limit`).
4. **Choose** between a plain LLM, RAG, from-scratch loop, and LangGraph for a
   given task, with a clear reason for the choice.

Tomorrow in **D4** we flip the problem: instead of building more on top of a
model, we make the model itself cheaper to run — fine-tune it on a task with
LoRA, then quantize it to 4-bit, and measure the memory and latency savings.

<div dir="rtl" align="right">

## الخلاصة

أربعة أمور تستطيع الآن:

١. تبني وكيل ReAct من الصفر في ~٦٠ سطرًا.
٢. تُعيد بناء نفس الوكيل كـ`StateGraph` في LangGraph بعُقَد وحوافّ وحافّة
   شرطيّة، وتعرف ما يقابل كلّ قطعة في النسخة من الصفر.
٣. تشرح لماذا نحتاج نموذجًا أكبر من D2 لثبات الصيغة، وترصد وضعَي الفشل
   (أداة مختلقة، حلقة لا نهائيّة) وصمّامات الأمان (`max_steps`، `recursion_limit`).
٤. تختار بين LLM عاري، RAG، حلقة من الصفر، وLangGraph لمهمّة معيّنة.

غدًا في **D4** نقلب الأمر: نُرخّص النموذج نفسه — ضبط دقيق بـLoRA ثم تكميم
٤-بت، وقياس الذاكرة والزمن.

</div>
